In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch
import joblib
import json
import hashlib
import numpy as np

RUN = Path(
    "/content/drive/MyDrive/MusicAnnotationProject/"
    "a5_contextual_safe_cleanup_v88/"
    "20261122_155756_notebook88_a5_training"
)

MODELS = RUN / "models"
CALIBRATORS = RUN / "calibrators"

OUT = RUN / "hf_export"
OUT.mkdir(exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# ============================================================
# 1. EXPORT CALIBRATORS
# ============================================================

calibrator_exports = []

for path in sorted(CALIBRATORS.glob("*.joblib")):

    obj = joblib.load(path)

    record = {
        "source_file": path.name,
        "sha256": sha256_file(path),
        "class": type(obj).__name__,
        "module": type(obj).__module__,
    }

    if hasattr(obj, "X_thresholds_"):
        record["X_thresholds"] = np.asarray(
            obj.X_thresholds_
        ).tolist()

    if hasattr(obj, "y_thresholds_"):
        record["y_thresholds"] = np.asarray(
            obj.y_thresholds_
        ).tolist()

    if hasattr(obj, "increasing_"):
        record["increasing"] = bool(obj.increasing_)

    calibrator_exports.append(record)

with open(OUT / "notebook88_calibrators.json", "w") as f:
    json.dump(calibrator_exports, f, indent=2)


# ============================================================
# 2. EXPORT MODEL STRUCTURE + WEIGHTS
# ============================================================

model_manifest = []

for path in sorted(MODELS.glob("*.pt")):

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    # Notebook 88 may have saved either a plain state_dict
    # or a wrapper dictionary.
    if isinstance(checkpoint, dict):

        if "state_dict" in checkpoint:
            state = checkpoint["state_dict"]

        elif "model_state_dict" in checkpoint:
            state = checkpoint["model_state_dict"]

        else:
            # Test whether this itself looks like a state_dict
            tensor_values = [
                isinstance(v, torch.Tensor)
                for v in checkpoint.values()
            ]

            if tensor_values and all(tensor_values):
                state = checkpoint
            else:
                state = None
    else:
        state = None

    record = {
        "source_file": path.name,
        "sha256": sha256_file(path),
        "checkpoint_type": str(type(checkpoint)),
    }

    if state is None:

        record["state_dict_found"] = False

        if isinstance(checkpoint, dict):
            record["checkpoint_keys"] = list(checkpoint.keys())

    else:

        record["state_dict_found"] = True

        weights = {}

        for name, tensor in state.items():

            arr = tensor.detach().cpu().numpy()

            weights[name] = {
                "shape": list(arr.shape),
                "dtype": str(arr.dtype),
                "values": arr.tolist(),
            }

        weight_file = (
            OUT /
            f"{path.stem}_weights.json"
        )

        with open(weight_file, "w") as f:
            json.dump(weights, f)

        record["weights_file"] = weight_file.name

        record["parameters"] = {
            name: {
                "shape": list(tensor.shape),
                "dtype": str(tensor.dtype),
                "numel": tensor.numel(),
            }
            for name, tensor in state.items()
        }

    model_manifest.append(record)

with open(OUT / "notebook88_models_manifest.json", "w") as f:
    json.dump(model_manifest, f, indent=2)


# ============================================================
# 3. SUMMARY
# ============================================================

summary = {
    "run": str(RUN),
    "models_found": len(model_manifest),
    "calibrators_found": len(calibrator_exports),
    "model_files": [
        x["source_file"] for x in model_manifest
    ],
    "calibrator_files": [
        x["source_file"] for x in calibrator_exports
    ],
}

with open(OUT / "notebook88_export_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print()
print("Export directory:", OUT)